# Cross-cell C2 · 冻结评估
在相同的 128 个 K562 留出扰动和每条件 300 个有序对照上配对比较 TxPert 与 PerturbDiff。

In [ ]:
import csv
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
runs = [(root.parent / "crosscell-c2-txpert", "TxPert"), (root, "PerturbDiff")]
pearson = (
    ("txpert_macro_pearson_delta_all", "TxPert Δ Pearson", "↑"),
    ("trishift_pearson_delta_all", "TriShift Δ Pearson", "↑"),
    ("systema_pearson_all", "Systema Pearson", "↑"),
)
metrics = [*pearson, ("mean_mse", "Mean MSE", "↓"), ("retrieval", "Retrieval", "↑")]
summaries, tables = [], []
for folder, name in runs:
    receipt = json.loads((folder / "result.json").read_text())
    raw = (folder / "conditions.jsonl").read_bytes()
    assert hashlib.sha256(raw).hexdigest() == receipt["conditions_file_sha256"]
    assert receipt["status"] == "evaluated" and receipt["formal_eligible"]
    assert receipt["test_updates"] == 0 and receipt["test_controls_per_condition"] == 300
    table = {}
    for line in raw.decode().splitlines():
        row = json.loads(line)
        assert row["condition_id"] not in table
        table[row["condition_id"]] = {
            **{m["metric_id"]: m["value"] for m in row["results"]},
            "mean_mse": row["mean_mse"], "retrieval": row["retrieval"],
        }
    assert len(table) == 128
    summaries.append(receipt)
    tables.append(table)
for identity in ("condition_ids_sha256", "paired_rows_sha256", "test_panel_genes",
                 "shared_fit_support_file_sha256"):
    assert summaries[0][identity] == summaries[1][identity], identity
assert list(tables[0]) == list(tables[1])
assert summaries[1]["test_panel_genes"] == 3352

colors = ("#566680", "#17666e")
fig, axes = plt.subplots(2, len(metrics), figsize=(13.4, 6.1), layout="constrained")
rng = np.random.default_rng(20)
columns = ["condition_id"] + [f"{name}__{metric}" for _, name in runs for metric, _, _ in metrics]
with (root / "condition-metrics.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(columns)
    for condition in tables[0]:
        writer.writerow([condition] + [
            tab[condition][metric] for tab in tables for metric, _, _ in metrics
        ])
for col, (metric, title, direction) in enumerate(metrics):
    low, high = axes[:, col]
    for pos, (table, receipt, color) in enumerate(zip(tables, summaries, colors, strict=True)):
        values = np.array([row[metric] for row in table.values()], dtype=float)
        values = values[np.isfinite(values)]
        if metric in {key for key, _, _ in pearson}:
            agg = next(item for item in receipt["pearson"] if item["metric_id"] == metric)
            assert len(values) == agg["finite_condition_count"]
            average = agg["macro_mean"]
        else:
            average = receipt[{"mean_mse": "mean_mse", "retrieval": "retrieval_mean"}[metric]]
        assert np.isclose(values.mean(), average), (metric, values.mean(), average)
        low.bar(pos, average, color=color, width=.62)
        low.text(pos, average, f"{average:.4f}", fontsize=8, ha="center",
                 va="bottom" if average >= 0 else "top")
        high.scatter(pos + rng.uniform(-.19, .19, len(values)), values, s=9, alpha=.5, color=color)
    low.set_title(f"{title} {direction}", fontsize=10)
    for ax in (low, high):
        ax.set_xticks([0, 1], ["TxPert", "PerturbDiff"], rotation=16)
        ax.set_xlim(-.5, 1.5)
        ax.margins(y=.22)
        ax.grid(axis="y", alpha=.2)
        ax.axhline(0, color="#333333", lw=.65)
axes[0,0].set_ylabel("Condition macro average")
axes[1,0].set_ylabel("Each frozen test condition")
fig.savefig(root / "frozen-metrics.png", dpi=175, bbox_inches="tight")
plt.show()